# BART Pipeline:
Input text -> Encoder -> Decoder -> Generated output

# Explanation:
BART is a sequence-to-sequence transformer model used for tasks such as
summarization and text generation.
Compared to DistilBART, BART is larger and usually slower,
but can produce richer summaries.

# Work:
Input text  
   ↓  
BART (Transformers + PyTorch)  
   ↓  
Summary / key content  
   ↓  
spaCy  
   ↓  
Entities: contributors, organizations, dates, etc.  

In [19]:
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import time
import re
import spacy

In [20]:
df = pd.read_json("../Data/silver/doc_01_silver_nlp.json")
print(df.head())

  document_id                                       cleaned_text  \
0      doc_01  Delta Water Innovation Hub\n\nProject ID: DWIH...   

                                              tokens  \
0  [delta, water, innovation, hub, project, id, s...   

                                           sentences  \
0  [Delta Water Innovation Hub\n\nProject ID: DWI...   

                                            entities  
0  [[Delta Water Innovation Hub\n\nProject ID, OR...  


In [21]:
BART_MODEL = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 35500.79it/s]


In [22]:
def summarize_bart(text,
                   max_input_len=1024,
                   max_output_len=200,
                   min_output_len=40):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [23]:
text = df.loc[0, "cleaned_text"]

summary = summarize_bart(text)
print(summary)

The Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland. The main goal is to reduce pressure on freshwater resources, improve water availability during drought periods and support circular water systems.


In [24]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


In [25]:
def evaluate_summary(original_text, summary_text, source_entities=None):
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)

    compression_ratio = summary_len / original_len if original_len > 0 else 0

    preserved_entities = []
    missed_entities = []

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    for ent in source_entities:
        ent_text = ent[0] if isinstance(ent, (list, tuple)) and len(ent) >= 1 else str(ent)
        ent_text_clean = " ".join(ent_text.split()).strip()

        if ent_text_clean and ent_text_clean.lower() in summary_lower:
            preserved_entities.append(ent_text_clean)
        else:
            missed_entities.append(ent_text_clean)

    total_entities = len(source_entities)
    entity_preservation_score = (
        len(preserved_entities) / total_entities if total_entities > 0 else None
    )

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }

In [26]:
def sentence_coverage_score(source_sentences, summary_text, threshold=0.2):
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue

        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0

In [27]:
start = time.time()
summary = summarize_bart(df.loc[0, "cleaned_text"])
runtime_seconds = time.time() - start

eval_result = evaluate_summary(
    original_text=df.loc[0, "cleaned_text"],
    summary_text=summary,
    source_entities=df.loc[0, "entities"]
)

eval_result["runtime_seconds"] = runtime_seconds
eval_result["sentence_coverage_score"] = sentence_coverage_score(
    df.loc[0, "sentences"], summary
)

print(summary)
print(eval_result)

The Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland. The main goal is to reduce pressure on freshwater resources, improve water availability during drought periods and support circular water systems.
{'original_token_count': 133, 'summary_token_count': 42, 'compression_ratio': 0.3157894736842105, 'preserved_entities': ['Zeeland'], 'missed_entities': ['Delta Water Innovation Hub Project ID', '2026-02-01', '2028-12-31', 'Lisa Vermeer', 'HZ Kenniscentrum Zeeuwse Samenleving and Water Technology', 'Lisa Vermeer', 'Tom de Bruin', 'Netherlands'], 'entity_preservation_score': 0.1111111111111111, 'runtime_seconds': 3.594053030014038, 'sentence_coverage_score': 1.0}


In [28]:
evaluation_row = {
    "document_id": df.loc[0, "document_id"],
    "model": "BART",
    "runtime_seconds": runtime_seconds,
    "original_token_count": eval_result["original_token_count"],
    "summary_token_count": eval_result["summary_token_count"],
    "compression_ratio": eval_result["compression_ratio"],
    "entity_preservation_score": eval_result["entity_preservation_score"],
    "sentence_coverage_score": eval_result["sentence_coverage_score"]
}

gold_eval_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "BART",
    "evaluation": evaluation_row
}

with open("../Data/gold/BART/BART_evaluation.json", "w") as f:
    json.dump(gold_eval_output, f, indent=4)

print("Saved BART evaluation to gold layer.")


Saved BART evaluation to gold layer.


In [29]:
nlp = spacy.load("en_core_web_sm")

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]

In [30]:
def extract_people(text):
    doc = nlp(text)
    people = [ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON"]
    return sorted(set(people))


In [31]:
def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text)
    return sorted(set(iso_dates))


In [32]:
def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text)
    return sorted(set(iso_dates))


In [33]:

def extract_description(summary, fallback_text):
    if summary and summary.strip():
        return summary.strip()
    return fallback_text[:500].strip()


In [37]:
def extract_title(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

In [34]:

def extract_research_groups(text, research_groups):
    text_lower = text.lower()
    matches = []

    for group in research_groups:
        if group.lower() in text_lower:
            matches.append(group)

    return sorted(set(matches))


In [35]:
def extract_project_metadata(text, summary=None, document_id=None):
    people = extract_people(text)
    dates = extract_dates(text)
    research_groups = extract_research_groups(text, RESEARCH_GROUPS)

    metadata = {
        "id": document_id or "",
        "title": extract_title(text),
        "description": extract_description(summary, text),
        "contact_person": people[0] if people else "",
        "start_date": dates[0] if len(dates) > 0 else "",
        "end_date": dates[1] if len(dates) > 1 else "",
        "research_groups": research_groups,
        "researchers": people
    }

    return metadata

In [38]:
metadata = extract_project_metadata(
    text=df.loc[0, "cleaned_text"],
    summary=summary,
    document_id=df.loc[0, "document_id"]
)

combined_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "BART",
    "summary": summary,
    "metadata": metadata,
    "evaluation": {
        "runtime_seconds": runtime_seconds,
        "original_token_count": eval_result["original_token_count"],
        "summary_token_count": eval_result["summary_token_count"],
        "compression_ratio": eval_result["compression_ratio"],
        "entity_preservation_score": eval_result["entity_preservation_score"],
        "sentence_coverage_score": eval_result["sentence_coverage_score"]
    }
}

with open("../Data/gold/BART/BART_output.json", "w") as f:
    json.dump(combined_output, f, indent=4)

print("Saved combined BART output to gold layer.")

Saved combined BART output to gold layer.
